# EGY-SCOUT V8 — Google Colab NVIDIA L4 / Kaggle T4 Full GPU Pipeline
**SigLIP + UMAP + 3-Class Shield + Realistic Referee Cap (<=4) + Coordinate Smoothing (<=36 km/h) + PARSeq Jersey OCR**

### Quick Steps in Colab:
1. Runtime -> Change runtime type -> **L4 GPU** (or **T4 GPU**).
2. Run **Cell 1** (install deps) & **Cell 2** (upload your match `.mp4` + optional `telemetry.parquet`).
3. Run **Cells 3 → 6** to get real GPU `SigLIP` team/referee separation + `PARSeq` jersey numbers + `Parquet` ready to drop into `D:\egy-scout-data\outputs\v8_3_parquet\`!


In [ ]:
# Cell 1: Detect GPU (L4/T4) & Install Packages
import os, time, json
from pathlib import Path
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU"
VRAM_GB = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2) if DEVICE == "cuda" else 0.0
BATCH_SIZE, PRECISION = (64, "bf16") if ("L4" in GPU_NAME.upper() or VRAM_GB >= 20) else ((32, "fp16") if DEVICE == "cuda" else (8, "fp32"))
print(f"[GPU] {GPU_NAME} ({VRAM_GB} GB) | Batch={BATCH_SIZE} | Precision={PRECISION}")

# Uncomment on a fresh Colab/Kaggle runtime:
# !pip install -q ultralytics supervision umap-learn transformers timm pyarrow pandas scikit-learn opencv-python-headless


In [ ]:
# Cell 2: Configure Inputs (Upload directly in Colab OR point to Google Drive)
MATCH_ID = "m2_ahly_enppi"
SPOTLIGHT_FRAME = 1250
RUN_PARSEQ_OCR = True   # Set False to skip jersey OCR and run even faster

OUT_DIR = Path("/content/egy_scout_outputs") if os.path.exists("/content") else Path("/kaggle/working/egy_scout_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Option A (Colab Direct Upload): uncomment to upload 9OcELOXQg2c.mp4 & telemetry.parquet
# from google.colab import files
# uploaded = files.upload()

# Option B (Google Drive): uncomment to mount Drive
# from google.colab import drive
# drive.mount('/content/drive')

VIDEO_PATH = Path("/content/9OcELOXQg2c.mp4")
LEGACY_PARQUET_PATH = Path("/content/telemetry.parquet")
print("Video:", VIDEO_PATH.is_file(), "| Legacy Parquet:", LEGACY_PARQUET_PATH.is_file())


In [ ]:
# Cell 3: Core Shield (4 Layers + Realistic Referee Cap <= 4 + Physical Smoothing <= 36 km/h + PARSeq OCR)
import re
from collections import defaultdict
import cv2
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from sklearn.cluster import KMeans

TRACKING_SCHEMA = pa.schema([
    ("match_id", pa.string()), ("frame_id", pa.int32()), ("time_s", pa.float32()),
    ("track_id", pa.int32()), ("role", pa.string()), ("team", pa.string()),
    ("role_conf", pa.float32()), ("team_conf", pa.float32()),
    ("x1", pa.float32()), ("y1", pa.float32()), ("x2", pa.float32()), ("y2", pa.float32()),
    ("u", pa.float32()), ("v", pa.float32()), ("x_m", pa.float32()), ("y_m", pa.float32()),
    ("on_pitch", pa.bool_()), ("jersey_number", pa.int16()), ("jersey_conf", pa.float32()),
    ("source_layer", pa.string()),
])


def smooth_coords(df: pd.DataFrame, max_kmh: float = 36.0) -> pd.DataFrame:
    out = df.copy()
    max_mps = max_kmh / 3.6
    for tid, idxs in out.groupby("track_id").groups.items():
        if int(tid) < 0 or len(idxs) < 3:
            continue
        grp = out.loc[idxs].sort_values("frame_id")
        ord_i = grp.index.to_numpy()
        xs = pd.Series(grp["x_m"].to_numpy(dtype=np.float64)).rolling(5, min_periods=1, center=True).median().to_numpy(copy=True)
        ys = pd.Series(grp["y_m"].to_numpy(dtype=np.float64)).rolling(5, min_periods=1, center=True).median().to_numpy(copy=True)
        ts = grp["time_s"].to_numpy(dtype=np.float64)
        for i in range(1, len(xs)):
            if not (np.isfinite(xs[i]) and np.isfinite(xs[i - 1])):
                continue
            dt = max(float(ts[i] - ts[i - 1]), 0.04)
            dist = float(np.hypot(xs[i] - xs[i - 1], ys[i] - ys[i - 1]))
            if dist > max_mps * dt and dist > 1e-6:
                r = (max_mps * dt) / dist
                xs[i] = xs[i - 1] + (xs[i] - xs[i - 1]) * r
                ys[i] = ys[i - 1] + (ys[i] - ys[i - 1]) * r
        out.loc[ord_i, "x_m"] = xs.astype(np.float32)
        out.loc[ord_i, "y_m"] = ys.astype(np.float32)
    return out


def extract_torso_crop(frame_bgr, bbox):
    h, w = frame_bgr.shape[:2]
    x1, y1, x2, y2 = [float(v) for v in bbox]
    bw, bh = max(1.0, x2 - x1), max(1.0, y2 - y1)
    tx1, tx2 = int(max(0, min(w - 1, round(x1 + 0.20 * bw)))), int(max(1, min(w, round(x1 + 0.80 * bw))))
    ty1, ty2 = int(max(0, min(h - 1, round(y1 + 0.15 * bh)))), int(max(1, min(h, round(y1 + 0.58 * bh))))
    crop = frame_bgr[ty1:ty2, tx1:tx2]
    return crop if (crop.shape[0] >= 10 and crop.shape[1] >= 10) else None


def calibrated_conf(points, centroids):
    dists = np.linalg.norm(points[:, None, :] - centroids[None, :, :], axis=2)
    order = np.argsort(dists, axis=1)
    labels = order[:, 0]
    d1, d2 = dists[np.arange(len(points)), order[:, 0]], dists[np.arange(len(points)), order[:, 1]]
    margin = np.clip((d2 - d1) / np.clip(d2, 1e-6, None), 0.0, 1.0)
    pair = [float(np.linalg.norm(centroids[i] - centroids[j])) for i in range(len(centroids)) for j in range(i + 1, len(centroids))]
    scale = max(float(np.median(pair)) * 0.45, 1e-4)
    logits = -(dists / scale)
    logits -= logits.max(axis=1, keepdims=True)
    probs = np.exp(logits) / np.clip(np.exp(logits).sum(axis=1, keepdims=True), 1e-8, None)
    return labels.astype(int), np.clip(0.5 * probs[np.arange(len(points)), labels] + 0.5 * margin, 0.0, 1.0).astype(np.float32)


def run_parseq_on_tracks(df: pd.DataFrame, crops: list) -> dict:
    # Legibility Gate + PARSeq Jersey OCR + Track-Level Weighted Voting
    if not RUN_PARSEQ_OCR:
        return {}
    try:
        from PIL import Image
        from torchvision import transforms as T
        parseq = torch.hub.load("baudm/parseq", "parseq", pretrained=True).eval().to(DEVICE)
        tf = T.Compose([T.Resize(parseq.hparams.img_size, T.InterpolationMode.BICUBIC), T.ToTensor(), T.Normalize(0.5, 0.5)])
    except Exception as exc:
        print("[WARN] PARSeq unavailable:", exc)
        return {}

    jersey_re = re.compile(r"\b([1-9][0-9]?)\b")
    track_obs = defaultdict(list)
    # Score crops by sharpness & height; read top 5 legible crops per track
    for tid, grp in df[df["role"].isin(["player", "goalkeeper"])].groupby("track_id"):
        cands = []
        for idx in grp.index:
            c = crops[idx]
            if c is None or c.shape[0] < 28 or c.shape[1] < 16:
                continue
            sharp = float(cv2.Laplacian(cv2.cvtColor(c, cv2.COLOR_BGR2GRAY), cv2.CV_64F).var())
            if sharp >= 35.0:
                cands.append((sharp * c.shape[0], c))
        cands.sort(key=lambda x: x[0], reverse=True)
        for _, crop in cands[:5]:
            pil = Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
            with torch.no_grad():
                logits = parseq(tf(pil).unsqueeze(0).to(DEVICE))
                labels, confs = parseq.tokenizer.decode(logits.softmax(-1))
            m = jersey_re.search(str(labels[0]))
            if m:
                cf = float(confs[0].mean().item()) if hasattr(confs[0], "mean") else float(confs[0])
                if cf >= 0.70:
                    track_obs[int(tid)].append((int(m.group(1)), cf))

    voted = {}
    for tid, obs in track_obs.items():
        scores = defaultdict(float)
        for num, cf in obs:
            scores[num] += cf
        best_n, best_w = max(scores.items(), key=lambda kv: kv[1])
        if best_w >= 1.2:
            voted[tid] = (best_n, round(min(0.99, best_w / (sum(scores.values()) or 1.0)), 3))
    return voted


In [ ]:
# Cell 4: Run Full Pipeline (SigLIP + UMAP + 3-Class Shield + PARSeq OCR) & Log GPU VRAM
import umap
from PIL import Image
from transformers import AutoProcessor, SiglipVisionModel

t0 = time.perf_counter()
if DEVICE == "cuda":
    torch.cuda.reset_peak_memory_stats()

cap = cv2.VideoCapture(str(VIDEO_PATH))
assert cap.isOpened(), f"Upload video first at {VIDEO_PATH}"
fps = float(cap.get(cv2.CAP_PROP_FPS)) or 25.0

if LEGACY_PARQUET_PATH.is_file():
    raw = pd.read_parquet(LEGACY_PARQUET_PATH)
    det_df = pd.DataFrame({
        "match_id": MATCH_ID, "frame_id": raw["frame_id"].astype("int32"),
        "time_s": raw.get("time_s", raw["frame_id"] / fps).astype("float32"),
        "track_id": raw["track_id"].astype("int32"),
        "role": raw.get("role", "player").astype(str), "team": "none",
        "role_conf": raw.get("conf", 0.85).astype("float32"), "team_conf": np.float32(0.0),
        "x1": raw["x1"].astype("float32"), "y1": raw["y1"].astype("float32"),
        "x2": raw["x2"].astype("float32"), "y2": raw["y2"].astype("float32"),
        "u": raw.get("u", (raw["x1"] + raw["x2"]) / 2.0).astype("float32"),
        "v": raw.get("v", raw["y2"]).astype("float32"),
        "x_m": raw.get("x_m", np.nan).astype("float32"), "y_m": raw.get("y_m", np.nan).astype("float32"),
        "on_pitch": raw.get("on_pitch", True).astype(bool),
        "jersey_number": np.int16(-1), "jersey_conf": np.float32(0.0), "source_layer": "input",
    })
    det_df = det_df[det_df["role"] != "ball"].reset_index(drop=True)
else:
    from ultralytics import YOLO
    yolo = YOLO("yolo11m.pt")
    rows, fid = [], 0
    while True:
        ok, fr = cap.read()
        if not ok:
            break
        if fid % 4 == 0:
            res = yolo.track(fr, persist=True, classes=[0], conf=0.35, tracker="bytetrack.yaml", verbose=False)[0]
            if res.boxes is not None and res.boxes.id is not None:
                for b, tid, cf in zip(res.boxes.xyxy.cpu().numpy(), res.boxes.id.int().cpu().numpy(), res.boxes.conf.cpu().numpy()):
                    rows.append({
                        "match_id": MATCH_ID, "frame_id": fid, "time_s": fid / fps, "track_id": int(tid),
                        "role": "player", "team": "none", "role_conf": float(cf), "team_conf": 0.0,
                        "x1": float(b[0]), "y1": float(b[1]), "x2": float(b[2]), "y2": float(b[3]),
                        "u": float((b[0] + b[2]) / 2), "v": float(b[3]), "x_m": np.nan, "y_m": np.nan,
                        "on_pitch": True, "jersey_number": -1, "jersey_conf": 0.0, "source_layer": "yolo11",
                    })
        fid += 1
    det_df = pd.DataFrame(rows)

det_df = smooth_coords(det_df, max_kmh=36.0)
crops = [None] * len(det_df)
for fid, grp in det_df.groupby("frame_id", sort=True):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(fid))
    ok, fr = cap.read()
    if ok:
        for idx, r in grp.iterrows():
            crops[idx] = extract_torso_crop(fr, [r["x1"], r["y1"], r["x2"], r["y2"]])
cap.release()

# SigLIP embeddings on L4/T4
valid_idx = [i for i, c in enumerate(crops) if c is not None]
pil_crops = [Image.fromarray(cv2.cvtColor(crops[i], cv2.COLOR_BGR2RGB)) for i in valid_idx]
proc = AutoProcessor.from_pretrained("google/siglip-base-patch16-224")
siglip = SiglipVisionModel.from_pretrained("google/siglip-base-patch16-224").to(DEVICE).eval()
amp_dt = torch.bfloat16 if PRECISION == "bf16" else (torch.float16 if PRECISION == "fp16" else torch.float32)

embs_list = []
with torch.no_grad():
    for i in range(0, len(pil_crops), BATCH_SIZE):
        inp = proc(images=pil_crops[i : i + BATCH_SIZE], return_tensors="pt").to(DEVICE)
        if DEVICE == "cuda" and PRECISION in ("bf16", "fp16"):
            with torch.amp.autocast("cuda", dtype=amp_dt):
                out = siglip(**inp)
        else:
            out = siglip(**inp)
        embs_list.append(torch.mean(out.last_hidden_state, dim=1).float().cpu().numpy())
embs = np.concatenate(embs_list, axis=0)

proj = umap.UMAP(n_components=3, n_neighbors=min(15, max(3, len(embs) - 1)), random_state=42).fit_transform(embs)
km = KMeans(n_clusters=3, n_init=10, random_state=42).fit(proj)
cids, confs = calibrated_conf(proj, km.cluster_centers_)
counts = sorted([(int((cids == k).sum()), k) for k in range(3)], reverse=True)
team_cids = sorted([counts[0][1], counts[1][1]])
cmap = {team_cids[0]: ("player", "team_a"), team_cids[1]: ("player", "team_b"), counts[2][1]: ("referee", "none")}

for idx, cid, cf in zip(valid_idx, cids, confs):
    r, t = cmap[int(cid)]
    if r == "player" and float(cf) < 0.65:
        det_df.at[idx, "role"], det_df.at[idx, "team"], det_df.at[idx, "role_conf"] = "referee", "none", 1.0 - float(cf)
    else:
        det_df.at[idx, "role"], det_df.at[idx, "team"], det_df.at[idx, "role_conf"], det_df.at[idx, "team_conf"] = r, t, float(cf), (float(cf) if t != "none" else 0.0)

# Layer 3 + Layer 4 Track Voting + Realistic Official Cap (<= 4) + Penalty-Box GK
main_refs, top_ars, bot_ars = [], [], []
for tid, idxs in det_df.groupby("track_id").groups.items():
    grp = det_df.loc[idxs]
    cal = grp[grp["y_m"].notna()]
    if not cal.empty:
        on_tl = ((cal["y_m"] >= -5.0) & (cal["y_m"] < 1.2)) | ((cal["y_m"] > 66.8) & (cal["y_m"] <= 73.0))
        if float(on_tl.mean()) >= 0.40:
            med_y = float(cal["y_m"].median())
            (top_ars if med_y < 34.0 else bot_ars).append((float(on_tl.mean()) * len(grp), int(tid), idxs))
            det_df.loc[idxs, ["role", "team", "on_pitch", "source_layer"]] = ["unassigned", "none", False, "layer4_tl_cand"]
            continue
    top_r = str(grp["role"].mode().iloc[0])
    if top_r == "referee":
        main_refs.append((float(len(grp)) * float(grp["role_conf"].mean()), int(tid), idxs))
        det_df.loc[idxs, ["role", "team", "source_layer"]] = ["unassigned", "none", "layer4_unassigned"]
    else:
        top_t = str(grp["team"].mode().iloc[0])
        det_df.loc[idxs, ["role", "team", "source_layer"]] = ["player", top_t, "layer4_temporal_vote"]

for _, _, idxs in sorted(main_refs, reverse=True)[:2]:
    det_df.loc[idxs, ["role", "team", "source_layer"]] = ["referee", "none", "layer4_verified_main_referee"]
for _, _, idxs in sorted(top_ars, reverse=True)[:1]:
    det_df.loc[idxs, ["role", "team", "on_pitch", "source_layer"]] = ["assistant_referee", "none", False, "layer4_verified_touchline_ar"]
for _, _, idxs in sorted(bot_ars, reverse=True)[:1]:
    det_df.loc[idxs, ["role", "team", "on_pitch", "source_layer"]] = ["assistant_referee", "none", False, "layer4_verified_touchline_ar"]

# PARSeq Jersey OCR on GPU
jersey_votes = run_parseq_on_tracks(det_df, crops)
for tid, (jnum, jconf) in jersey_votes.items():
    det_df.loc[det_df["track_id"] == tid, "jersey_number"] = np.int16(jnum)
    det_df.loc[det_df["track_id"] == tid, "jersey_conf"] = np.float32(jconf)

elapsed = round(time.perf_counter() - t0, 2)
peak_vram_mb = round(torch.cuda.max_memory_allocated() / (1024**2), 1) if DEVICE == "cuda" else 0.0
print(f"[DONE] Time: {elapsed}s | Peak VRAM: {peak_vram_mb} MB | Jersey numbers read: {len(jersey_votes)}")


In [ ]:
# Cell 5: Export Apache Parquet (Zstd) + Summary JSON (Drop directly into D:\egy-scout-data\outputs\v8_3_parquet\)
pq_out = OUT_DIR / f"{MATCH_ID}_v8.parquet"
cols = [f.name for f in TRACKING_SCHEMA]
pq.write_table(pa.Table.from_pandas(det_df[cols], schema=TRACKING_SCHEMA, preserve_index=False), str(pq_out), compression="zstd")

summary = {
    "match_id": MATCH_ID, "gpu": GPU_NAME, "elapsed_s": elapsed, "peak_vram_mb": peak_vram_mb,
    "unique_officials": int(det_df[det_df["role"].isin(["referee", "assistant_referee"])]["track_id"].nunique()),
    "jerseys_detected": {str(k): {"number": v[0], "conf": v[1]} for k, v in jersey_votes.items()},
}
sum_out = OUT_DIR / f"{MATCH_ID}_summary.json"
sum_out.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print("[SAVED]", pq_out, "|", sum_out)
print(json.dumps(summary, indent=2))

# Uncomment on Colab to auto-download Parquet straight to your PC:
# from google.colab import files
# files.download(str(pq_out))
